In [ ]:
# Import thu vien
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score

In [ ]:
# Doc du lieu train, valid, test
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(processed_dir / "train.csv")
valid_df = pd.read_csv(processed_dir / "valid.csv")
test_df = pd.read_csv(processed_dir / "test.csv")
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test:", test_df.shape)
train_df.head()

In [ ]:
# Cau hinh target va threshold
target_cols = ["rain_next_1h", "rain_next_2h", "rain_next_3h"]
threshold = 0.5

target_cols

In [ ]:
# Tao feature sin cos cho bien chu ky
def add_cyclic_features(df):
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)
    df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["wind_dir_sin"] = np.sin(2 * np.pi * df["wind_direction_10m"] / 360)
    df["wind_dir_cos"] = np.cos(2 * np.pi * df["wind_direction_10m"] / 360)
    return df

train_df = add_cyclic_features(train_df)
valid_df = add_cyclic_features(valid_df)
test_df = add_cyclic_features(test_df)
train_valid_df = add_cyclic_features(train_valid_df)

drop_cols = ["datetime", "hour", "month", "dayofweek", "wind_direction_10m"] + target_cols
feature_cols = [col for col in train_df.columns if col not in drop_cols]

print("num_features:", len(feature_cols))
feature_cols

In [ ]:
# Tach feature dau vao
X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]
X_train_valid = train_valid_df[feature_cols]

X_train.shape, X_valid.shape, X_test.shape

In [ ]:
# Ham tao model va ham danh gia
def make_model(params):
    return RandomForestClassifier(
        **params,
        bootstrap=True,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )


def evaluate_one_target(y_true, y_prob, y_pred, target, split_name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "model": "random_forest",
        "split": split_name,
        "target": target,
        "threshold": threshold,
        "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

In [ ]:
# Tune tham so tren valid
param_grid = [
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 3},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 3},
    {"n_estimators": 400, "max_depth": 16, "min_samples_leaf": 5},
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 5},
]

tuning_rows = []
best_params_by_target = {}

for target in target_cols:
    best_score = float("inf")
    best_params = None

    for params in param_grid:
        print("Target:", target, "Params:", params)
        model = make_model(params)
        model.fit(X_train, train_df[target])

        valid_prob = model.predict_proba(X_valid)[:, 1]
        valid_pred = (valid_prob >= threshold).astype(int)
        metric = evaluate_one_target(valid_df[target], valid_prob, valid_pred, target, "valid")
        metric.update(params)
        tuning_rows.append(metric)

        if metric["log_loss"] < best_score:
            best_score = metric["log_loss"]
            best_params = params

    best_params_by_target[target] = best_params

tuning_df = pd.DataFrame(tuning_rows)
best_params_df = pd.DataFrame([
    {"target": target, **params}
    for target, params in best_params_by_target.items()
])

tuning_df.to_csv(results_dir / "random_forest_tuning_results.csv", index=False)

best_params_df

In [ ]:
# Train final tren train valid va test final
final_models = {}
test_metric_rows = []
test_predictions = test_df[["datetime"]].copy()

for target in target_cols:
    params = best_params_by_target[target]
    print("Final target:", target, "Params:", params)

    model = make_model(params)
    model.fit(X_train_valid, train_valid_df[target])
    final_models[target] = model

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)
    metric = evaluate_one_target(test_df[target], test_prob, test_pred, target, "test")
    metric.update(params)
    test_metric_rows.append(metric)

    test_predictions[f"{target}_true"] = test_df[target].values
    test_predictions[f"{target}_prob"] = test_prob
    test_predictions[f"{target}_pred"] = test_pred

metrics_df = pd.DataFrame(test_metric_rows)
metrics_df = metrics_df[[
    "model", "split", "target", "threshold", "log_loss", "brier_score", "roc_auc", "pr_auc",
    "accuracy", "precision", "recall", "f1", "tn", "fp", "fn", "tp",
]]

metrics_df.to_csv(results_dir / "random_forest_test_metrics.csv", index=False)
test_predictions.to_csv(results_dir / "random_forest_test_predictions.csv", index=False)

metrics_df

In [ ]:
# Xem feature quan trong nhat cua Random Forest
feature_importance_rows = []

for target, model in final_models.items():
    for feature, importance in zip(feature_cols, model.feature_importances_):
        feature_importance_rows.append({
            "target": target,
            "feature": feature,
            "importance": importance,
        })

feature_importance_df = pd.DataFrame(feature_importance_rows)
feature_importance_df.sort_values(["target", "importance"], ascending=[True, False]).groupby("target").head(10)